In [3]:
import numpy as np
from sklearn.linear_model import LogisticRegression

## Logistic Regression Scratch

In [4]:
class LogisticRegressionOwn():
    def __init__(self,learning_rate =0.01,epoch = 1000,tol = 1e-5,patience =1000):
        self.learning_rate = learning_rate
        self.epoch =epoch
        self.tol = tol
        self.patience =patience
        self.weights = None
        self.bias = 0.0
    def init_parameters(self,n):
        self.weights = np.zeros((n,1))
        self.bias =0.0

    def sigmoid(self,z):
        return 1/(1+np.exp(-z))

    def predict_proba(self,X):
        z =X @ self.weights + self.bias
        return self.sigmoid(z)
    def predict(self,X,threshold):
        p = self.predict_proba(X)
        return (p>=threshold).astype(int)
    def compute_gradients(self,X,p,y):
        m = y.shape[0]
        dw = (X.T @ (p-y)) / m
        db = np.mean(p-y)
        return dw,db
    def update_parameters(self,dw,db):
        self.weights -= self.learning_rate*dw
        self.bias -= self.learning_rate*db
    def compute_cost(self, y, p):
        eps = 1e-15
        p = np.clip(p, eps, 1 - eps)
        cost = -np.mean(
            y * np.log(p) +
            (1 - y) * np.log(1 - p)
        )
        return cost
    def fit(self,X,y):
        if X.shape[0] != y.shape[0]:
            raise ValueError("X,y length mismatch")
        m,n = X.shape
        self.init_parameters(n)
        prev_cost =None
        count = 0
        for _ in range(self.epoch):
            p = self.predict_proba(X)
            dw,db = self.compute_gradients(X,p,y)
            self.update_parameters(dw,db)
            cost =self.compute_cost(y,p)
            if prev_cost is not None and abs(cost-prev_cost) < self.tol:
                if count < self.patience:
                    count+=1
                else:
                    print("model converged")
                    break
            else:
                count=0
            prev_cost=cost
        return self
            
            
        
        
        
        

## OneVSRest

In [7]:
from copy import deepcopy

class OneVsRest():
    def __init__(self,learning_rate =0.01,epoch =1000,tol =1e-10,patience =1000,estimator=None):
        if estimator is None:
                estimator = LogisticRegressionOwn(
                learning_rate=learning_rate,
                epoch=epoch,
                tol=tol,
                patience=patience
    )
        self.estimator = estimator
        self.classes = None
        self.models = dict()
    def fit(self,X,y):
        self.classes =np.unique(y)
        for cls in self.classes:
            y_bin = (y==cls).astype(int)
            model = deepcopy(self.estimator)
            model.fit(X,y_bin)
            self.models[cls] =model
        return self
    def predict_proba(self,X):
        proba = []
        for cls in self.classes:
            cur_proba= self.models[cls].predict_proba(X)
            proba.append(cur_proba)
        return np.hstack(proba)
    def predict(self,X):
        proba = self.predict_proba(X)
        indices = np.argmax(proba,axis=1)
        return self.classes[indices]


            
        
        
        

## Softmax Regression

In [12]:
class SoftMaxRegression():
    def __init__(self,learning_rate =0.01,epoch=1000,tol=1e-5,patience =1000):
        self.learning_rate =learning_rate
        self.epoch =epoch
        self.tol = tol
        self.patience =patience
        self.classes = None
        self.n_classes = None
    def one_hot(self, y):
        one_hot = np.zeros((len(y), self.n_classes))
        self.class_to_index = {
    c: i for i, c in enumerate(self.classes)
}

        indices = np.array([self.class_to_index[c] for c in y])
        one_hot[np.arange(len(y)), indices] = 1
        return one_hot
    def init_parameters(self,n):
        self.weights = np.zeros((n,self.n_classes))
        self.bias = np.zeros((1,self.n_classes))
    def compute_gradients(self,X,y,p):
        dw = X.T @ (p-y)/X.shape[0]
        db = np.sum(p - y, axis=0, keepdims=True) / X.shape[0]        
        return dw,db
    def softmax(self, z):
        # Numerical stability
        z = z - np.max(z, axis=1, keepdims=True)
        exp_z = np.exp(z)
        return exp_z / np.sum(exp_z, axis=1, keepdims=True)   
    def predict_proba(self,X):
        z = X @ self.weights + self.bias
        return self.softmax(z)
    def predict(self,X):
        p = self.predict_proba(X)
        idx =np.argmax(p, axis=1)
        return self.classes[idx]
    def update_gradients(self,dw,db):
        self.weights = self.weights - self.learning_rate*dw
        self.bias = self.bias - self.learning_rate*db
    def compute_loss(self, y_true, y_pred):
        m = y_true.shape[0]
        return -np.sum(y_true * np.log(y_pred + 1e-15)) / m
    def fit(self,X,y):
        m,n = X.shape
        self.classes = np.unique(y)
        self.n_classes = len(self.classes)
        y = self.one_hot(y)
        prev_cost=None
        count = 0
        self.init_parameters(n)
        for _ in range(self.epoch):
            prob = self.predict_proba(X)

            cost = self.compute_loss(y,prob)
            if prev_cost is not None and abs(cost-prev_cost) < self.tol:
                if count < self.patience:
                    count+=1
                else:
                    print("model converged")
                    break
            else:
                count=0
            prev_cost=cost
            dw,db = self.compute_gradients(X,y,prob)
            self.update_gradients(dw,db)
        return self
            
            ## forward
        
        
                
                
        